In [1]:
%pip install -q plotly

In [2]:
import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display, HTML
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
# Upload of the CSV files
uploaded = files.upload()

# Obtain the uploaded file name
filename = next(iter(uploaded))

# Read data and save in a dataframe
df = pd.read_csv(filename)

Saving owid-co2-data.csv to owid-co2-data.csv


In [4]:
# Show the main information of the dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50411 entries, 0 to 50410
Data columns (total 79 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   country                                    50411 non-null  object 
 1   year                                       50411 non-null  int64  
 2   iso_code                                   42480 non-null  object 
 3   population                                 41167 non-null  float64
 4   gdp                                        15251 non-null  float64
 5   cement_co2                                 29173 non-null  float64
 6   cement_co2_per_capita                      25648 non-null  float64
 7   co2                                        29384 non-null  float64
 8   co2_growth_abs                             27216 non-null  float64
 9   co2_growth_prct                            26239 non-null  float64
 10  co2_including_luc     

In [5]:
# Filtering of the required columns
required_columns = [
    "country",
    "year",
    "iso_code",
    "co2",
    "co2_per_capita"
]

# ValueError raised if not found on the dataframe
missing = [col for col in required_columns if col not in df.columns]
if missing:
  raise ValueError(f"Missing required columns in dataset: {missing}")

# Dataset Relevant Information:
print(f"Year ranges: {df["year"].min()} to {df["year"].max()}")
print(f"Number of observations: {len(df)}")
print(f"Number of observations with ISO Code missing: "
    f"{df["iso_code"].isna().sum()}")
print(f"Number of observations with CO2 missing: {df['co2'].isna().sum()}")
print(f"Number of observations with CO2 per capita missing: "
    f"{df['co2'].isna().sum()}")

Year ranges: 1750 to 2024
Number of observations: 50411
Number of observations with ISO Code missing: 7931
Number of observations with CO2 missing: 21027
Number of observations with CO2 per capita missing: 21027


In [6]:
# Exclusde registers withou ISO Code, which do not represent actual countries
df_countries = df[
    df["iso_code"].notna()
    & df["iso_code"].astype(str).str.strip().ne("")
].copy()

# Show metrics afte rfilter
print(f"Original Num Observations: {len(df)}")
print(f"Number of Observations after Filter: {len(df_countries)}")
print(f"Number of countries with ID: {df_countries['iso_code'].nunique()}")

# Show some examples of data observations
df_no_countries = df[
    df["iso_code"].isna()
    | df["iso_code"].astype(str).str.strip().eq("")
]

display(df_no_countries[["country", "year", "iso_code"]]
    .drop_duplicates()
    .head(10)
)

Original Num Observations: 50411
Number of Observations after Filter: 42480
Number of countries with ID: 218


,country,year,iso_code
275,Africa,1750,NaN
276,Africa,1751,NaN
277,Africa,1752,NaN
278,Africa,1753,NaN
279,Africa,1754,NaN
280,Africa,1755,NaN
281,Africa,1756,NaN
282,Africa,1757,NaN
283,Africa,1758,NaN
284,Africa,1759,NaN


In [7]:
# Code to filter the 15 countries with a higher number of overall CO2
# emissions which will be the countries represented in the horizon chart
df_2024 = df_countries[
    (df_countries["year"].between(1950, 2024))
].copy()

# Identify countries with exactly one observation per year and strictly
# positive values for both co2 and co2 per capita indicators in the 75 years
country_quality = (
    df_2024.groupby(["country", "iso_code"])
    .agg(
        n_years=("year", "nunique"),
        co2_valid=("co2", lambda s: s.notna().all() and (s > 0).all()),
        co2_per_capita_valid=(
            "co2_per_capita",
            lambda s: s.notna().all() and (s > 0).all()
        ),
        first_year=("year", "min"),
        last_year=("year", "max")
    )
    .reset_index()
)

eligible_countries = country_quality[
    (country_quality["n_years"] == 75)
    & (country_quality["first_year"] == 1950)
    & (country_quality["last_year"] == 2024)
    & country_quality["co2_valid"]
    & country_quality["co2_per_capita_valid"]
]["country"].tolist()

# Filter year 2024 and retain only eligible countries
df_2024 = df_countries[
    (df_countries["year"] == 2024)
    & (df_countries["co2"].notna())
    & (df_countries["co2"] > 0)
    & (df_countries["co2_per_capita"].notna())
    & (df_countries["co2_per_capita"] > 0)
    & (df_countries["country"].isin(eligible_countries))
].copy()

# Code to verify no duplicates are on the new df_2024
duplicates_2024 = df_2024["country"].duplicated().sum()
print(f"Duplicated Countries in 2024 dataframe: {duplicates_2024}")

# Raise Value error if duplicates found
if duplicates_2024 > 0:
  raise ValueError("Duplication of countries on year 2024")

# Descending order of the countries with higher CO2 emissions
ranking_2024 = (
    df_2024
    .sort_values("co2", ascending=False)
    .head(15).
    copy()
)

# Generate a list with the countries identifiers for the horizon chart
selected_countries = ranking_2024["country"].tolist()
selected_countries_iso = ranking_2024["iso_code"].tolist()

# Display the selected countries
display(
    ranking_2024[["country", "year", "iso_code", "co2", "co2_per_capita"]]
)

Duplicated Countries in 2024 dataframe: 0


,country,year,iso_code,co2,co2_per_capita
9933,China,2024,CHN,12289.037109,8.658390
48007,United States,2024,USA,4904.119629,14.197287
21762,India,2024,IND,3193.478027,2.200978
38314,Russia,2024,RUS,1780.524414,12.294705
23812,Japan,2024,JPN,961.867310,7.772474
21937,Indonesia,2024,IDN,812.220215,2.865096
22662,Iran,2024,IRN,792.631165,8.656227
43038,South Korea,2024,KOR,583.679138,11.285893
18319,Germany,2024,DEU,572.319153,6.768824
8877,Canada,2024,CAN,533.340088,13.419917


In [8]:
# Prepare the dataset for the Horizon Graph for the selected 15 countries
df_horizon = df_countries[
    df_countries["iso_code"].isin(selected_countries_iso)
    & df_countries["year"].between(1950, 2024)
].copy()

# Leave only the required columns for the horizon chart
df_horizon = df_horizon[
    ["country", "iso_code", "year", "co2", "co2_per_capita"]
].sort_values(["iso_code", "year"])

# Chech the range and potential missing values
summary_df_horizon = (
    df_horizon.groupby(["country", "iso_code"])
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        num_registers=("year", "nunique"),
        values_per_capita=("co2_per_capita", "count"),
        missing_values_per_capita=("co2_per_capita", lambda s: s.isna().sum())
    )
    .reset_index()
    .sort_values("country")
)

display(summary_df_horizon)

print(f"Selected countries: {df_horizon['iso_code'].nunique()}")
print(f"Rows for the period: {len(df_horizon):,}")
print(f"Years per country: 75")


,country,iso_code,first_year,last_year,num_registers,values_per_capita,missing_values_per_capita
0,Australia,AUS,1950,2024,75,75,0
1,Brazil,BRA,1950,2024,75,75,0
2,Canada,CAN,1950,2024,75,75,0
3,China,CHN,1950,2024,75,75,0
4,Germany,DEU,1950,2024,75,75,0
5,India,IND,1950,2024,75,75,0
6,Indonesia,IDN,1950,2024,75,75,0
7,Iran,IRN,1950,2024,75,75,0
8,Japan,JPN,1950,2024,75,75,0
9,Mexico,MEX,1950,2024,75,75,0


Selected countries: 15
Rows for the period: 1,125
Years per country: 75


In [9]:
# Dataframe copy from original
df_horizon = df_horizon.sort_values(
    ["iso_code", "year"]
).copy()

# Calculate metrics for each country
group = df_horizon.groupby("iso_code")["co2_per_capita"]

# Last year value
df_horizon["co2_per_capita_prev"] = group.shift(1)

# Verify last year is consecutive
df_horizon["year_prev"] = (
    df_horizon.groupby("iso_code")["year"].shift(1)
)

consecutive = df_horizon["year"] == df_horizon["year_prev"] + 1

# Define Absolute CO2 per capita Change
df_horizon["absolute_change"] = (
    df_horizon["co2_per_capita"]
    - df_horizon["co2_per_capita_prev"]
).where(consecutive)

# Define Relative CO2 per capita Change
df_horizon["percentual_change"] = (
    df_horizon["absolute_change"]
    / df_horizon["co2_per_capita_prev"].replace(0, np.nan)
) * 100

# Revisar las primeras observaciones de cada país
display(
    df_horizon[
        [
            "country",
            "year",
            "co2_per_capita",
            "co2_per_capita_prev",
            "absolute_change",
            "percentual_change"
        ]
    ].groupby("country", group_keys=False).head(3)
)


,country,year,co2_per_capita,co2_per_capita_prev,absolute_change,percentual_change
3604,Australia,1950,6.804118,NaN,NaN,NaN
3605,Australia,1951,7.251952,6.804118,0.447834,6.581801
3606,Australia,1952,7.230732,7.251952,-0.021220,-0.292607
7138,Brazil,1950,0.368088,NaN,NaN,NaN
7139,Brazil,1951,0.393671,0.368088,0.025583,6.950215
7140,Brazil,1952,0.439798,0.393671,0.046127,11.717107
8803,Canada,1950,11.216469,NaN,NaN,NaN
8804,Canada,1951,11.551389,11.216469,0.334920,2.985966
8805,Canada,1952,11.006766,11.551389,-0.544622,-4.714779
9859,China,1950,0.144906,NaN,NaN,NaN


In [10]:

# Summary of the two calculated metrics
display(
    df_horizon[
        ["absolute_change", "percentual_change"]
    ].describe().round(3)
)

# Count increases, decreases and unchanged observations
change = df_horizon["absolute_change"]

print(f"Increases: {(change > 0).sum()}")
print(f"Decreases: {(change < 0).sum()}")
print(f"No change: {(change == 0).sum()}")
print(f"Undefined annual changes: {change.isna().sum()}")

# Check for unexpected missing values in the selected data
print(
    "Missing absolute changes after the first year:",
    df_horizon.loc[df_horizon["year"] > 1950, "absolute_change"].isna().sum()
)


,absolute_change,percentual_change
count,1110.000,1110.000
mean,0.060,3.711
std,0.351,41.932
min,-2.925,-73.403
25%,-0.050,-1.023
50%,0.048,1.874
75%,0.204,5.341
max,1.317,1376.281


Increases: 740
Decreases: 370
No change: 0
Undefined annual changes: 15
Missing absolute changes after the first year: 0


In [11]:
df_horizon.head()

,country,iso_code,year,co2,co2_per_capita,co2_per_capita_prev,year_prev,absolute_change,percentual_change
3604,Australia,AUS,1950,55.635578,6.804118,NaN,NaN,NaN,NaN
3605,Australia,AUS,1951,61.051395,7.251952,6.804118,1950.0,0.447834,6.581801
3606,Australia,AUS,1952,62.406326,7.230732,7.251952,1951.0,-0.021220,-0.292607
3607,Australia,AUS,1953,63.002892,7.146357,7.230732,1952.0,-0.084375,-1.166893
3608,Australia,AUS,1954,67.920609,7.547251,7.146357,1953.0,0.400894,5.609763


In [12]:
df_horizon.tail()

,country,iso_code,year,co2,co2_per_capita,co2_per_capita_prev,year_prev,absolute_change,percentual_change
42409,South Africa,ZAF,2020,435.297668,7.187593,7.899718,2019.0,-0.712125,-9.014566
42410,South Africa,ZAF,2021,439.464386,7.145460,7.187593,2020.0,-0.042132,-0.586182
42411,South Africa,ZAF,2022,428.782166,6.873887,7.145460,2021.0,-0.271574,-3.800645
42412,South Africa,ZAF,2023,436.604187,6.906940,6.873887,2022.0,0.033054,0.480862
42413,South Africa,ZAF,2024,439.830505,6.871580,6.906940,2023.0,-0.035361,-0.511961


In [13]:
# Configuration of 4 bands to generate the horizon graph
number_bands = 4

# Country order of appearance based on descending CO2 emissions in 2024
country_order = ranking_2024["country"].tolist()

# Colors for emissions light to dark
emission_colors = [
    "#DCEAF7",
    "#91BCE4",
    "#4388C7",
    "#174A7E"
]

# Code to generate a function add_filled_band to draw the band between a
# baseline and the upper boundary
# Reference: https://anyplot.ai/horizon-basic/python/plotly
# Reference 2: https://community.plotly.com/t/stacked-horizon-plots/57838
# Reference 3: ChatGPT Prompt: "Review the code to support a horizon chart to
# represent the CO2 emissions per capita for 15 countries in a cronological
# order using blue colours"
def add_filled_band(fig, x, y_base, y_top, color, name,
                    row, col=1, showlegend=False):
    """Draw one filled band between a baseline and its upper boundary"""
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y_base,
            mode="lines",
            line=dict(width=0, color=color),
            hoverinfo="skip",
            showlegend=False
        ),
        row=row, col=col
    )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y_top,
            mode="lines",
            line=dict(width=0, color=color),
            fill="tonexty",
            fillcolor=color,
            name=name,
            showlegend=showlegend,
            hoverinfo="skip"
        ),
        row=row, col=col
    )


In [14]:
# Code to generate the horizon graph with the build_emissions_horizon function
# Reference: https://anyplot.ai/horizon-basic/python/plotly
# Reference 2: https://community.plotly.com/t/stacked-horizon-plots/57838
# Reference 3: ChatGPT Prompt: "Review the code to support a horizon chart to
# represent the CO2 emissions per capita for 15 countries in a cronological
# order using blue colours"

def build_emissions_horizon(data):
    """Horizon Graph with compact layout and improved legend."""

    n_countries = len(country_order)

    # Code to configure subplots to the fig object
    # Reference: https://plotly.com/python-api-reference/generated/
    # plotly.subplots.make_subplots.html
    fig = make_subplots(
        rows=n_countries,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.004
    )

    # Define the common scale across all countries and for the bands
    maximum = data["co2_per_capita"].max()
    band_width = maximum / number_bands

    # Order the data cronologically and obtain "year" and "co2_per_capita"
    for row, country in enumerate(country_order, start=1):
        series = (
            data[data["country"] == country]
            .sort_values("year")
        )

        x = series["year"].to_numpy()
        values = series["co2_per_capita"].to_numpy()

        # For each band draw its magnitude using add_filled_band function
        for band in range(number_bands):
            lower = band * band_width
            band_values = np.clip(values - lower, 0, band_width)

            add_filled_band(
                fig=fig,
                x=x,
                y_base=np.zeros(len(values)),
                y_top=band_values,
                color=emission_colors[band],
                name=f"Band {band + 1}",
                row=row
            )

        # Interactive layer to obtain the exact values
        # Reference: https://plotly.com/python-api-reference/generated/
        # generated/plotly.graph_objects.Figure.add_traces.html
        fig.add_trace(
            go.Scatter(
                x=x,
                y=np.full(len(x), band_width * 0.5),
                mode="markers",
                marker=dict(
                    size=12,
                    color="rgba(0,0,0,0.01)"
                ),
                customdata=np.column_stack([
                    series["country"].to_numpy(),
                    values
                ]),
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Año: %{x}<br>"
                    "Emisiones: %{customdata[1]:.2f} "
                    "toneladas de CO₂ por persona y año"
                    "<extra></extra>"
                ),
                showlegend=False,
                name=country
            ),
            row=row,
            col=1
        )

        # Code to add a rectangular shape to each plot to distinguish the
        # differences clearly
        fig.add_shape(
            type="rect",
            x0=1950,
            x1=2024,
            y0=0,
            y1=band_width,
            xref=f"x{row} domain" if False else "x",
            yref=f"y{row}" if row > 1 else "y",
            line=dict(
                color="#D9DEE5",
                width=0.7
            ),
            fillcolor="rgba(0,0,0,0)",
            layer="above",
            row=row,
            col=1
        )

        # Code to configure teh position and dimensions of country labels
        fig.add_annotation(
            x=-0.018,
            y=1 - (row - 0.5) / n_countries,
            xref="paper",
            yref="paper",
            text=country,
            showarrow=False,
            xanchor="right",
            yanchor="middle",
            font=dict(size=13, color="#263238")
        )

    # Code to set up the legend in ascending order, this is lightest to darkest
    for band in range(number_bands):
        lower = band * band_width
        upper = (band + 1) * band_width

        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(
                    size=10,
                    symbol="square",
                    color=emission_colors[band]
                ),
                name=f"{lower:.1f} – {upper:.1f} t CO₂/person/year",
                hoverinfo="skip",
                showlegend=True
            )
        )

    # Code to hide the vertical axes
    fig.update_yaxes(
        visible=False,
        range=[0, band_width],
        fixedrange=True
    )

    # Code to ensure shared time axis and title and year labels only at the
    # bottom of the graph
    for row in range(1, n_countries + 1):
        fig.update_xaxes(
            row=row,
            col=1,
            range=[1950, 2024],
            dtick=5,
            showgrid=True,
            gridcolor="#E5E7EB",
            showticklabels=(row == n_countries),
            title_text="Año" if row == n_countries else None,
            showspikes=True,
            spikemode="across",
            spikesnap="cursor",
            spikecolor="#333333",
            spikethickness=1,
            spikedash="solid"
        )

    # Code to set up the global y-axis title
    fig.add_annotation(
        x=-0.14,
        y=0.5,
        xref="paper",
        yref="paper",
        text="Toneladas de CO₂ por persona y año",
        textangle=-90,
        showarrow=False,
        font=dict(size=13)
    )

    # Code to add the source of the data
    fig.add_annotation(
        x=0,
        y=-0.1,
        xref="paper",
        yref="paper",
        text="Fuente: Our World in Data - CO₂ and Greenhouse Gas Emissions Data"
        " Explorer (owid-co2-data.csv) Link: https://github.com/owid/co2-data",
        showarrow=False,
        xanchor="left",
        yanchor="top",
        font=dict(size=11)
    )

    # Code to add the author of the visualization
    fig.add_annotation(
        x=0,
        y=-0.15,
        xref="paper",
        yref="paper",
        text="Autor: Victor Calvo Serra",
        showarrow=False,
        xanchor="left",
        yanchor="top",
        font=dict(size=11)
    )

    fig.update_layout(
        title=dict(
            text=(
                "Horizon Chart - Comparativa de emisiones "
                "de CO₂ per cápita entre 1950 y 2024"
            ),
            x=0.5,
            xanchor="center",
            font=dict(size=21)
        ),
        height=700,
        template="plotly_white",
        margin=dict(l=190, r=25, t=100, b=100),
        hovermode="closest",
        hoverdistance=30,
        spikedistance=-1,
        legend=dict(
            title="Intervalos de emisiones",
            orientation="h",
            yanchor="bottom",
            y=1.01,
            xanchor="center",
            x=0.5,
            font=dict(size=9),
            traceorder="normal"
        )
    )

    return fig


# Code to build the chart with the dataframe df_horizon and display the
# chart
fig_emissions = build_emissions_horizon(df_horizon)
fig_emissions.show()


In [15]:
# Code to get the reference data for 2024
data_2024 = df_horizon[df_horizon["year"] == 2024].copy()

# Code to sort countries by per-capita emissions
highest_per_capita = data_2024.nlargest(3, "co2_per_capita")
lowest_per_capita = data_2024.nsmallest(3, "co2_per_capita")

# Code to get the first and last available observations in the selected period
first_year = df_horizon["year"].min()
last_year = df_horizon["year"].max()

data_first = df_horizon[df_horizon["year"] == first_year].set_index("country")
data_last = df_horizon[df_horizon["year"] == last_year].set_index("country")

# Code to calculate the absolute change between the first and last years
period_change = (
    data_last["co2_per_capita"] - data_first["co2_per_capita"]
).sort_values(ascending=False)

# Code to buuild a short summary
highest_text = ", ".join(
    f"{row['country']} ({row['co2_per_capita']:.2f})"
    for _, row in highest_per_capita.iterrows()
)

lowest_text = ", ".join(
    f"{row['country']} ({row['co2_per_capita']:.2f})"
    for _, row in lowest_per_capita.iterrows()
)

largest_increase_country = period_change.index[0]
largest_increase = period_change.iloc[0]

largest_decrease_country = period_change.index[-1]
largest_decrease = period_change.iloc[-1]

# Reference 1: ChatGPT Prompt: "Review the code to display html comments
# for a description to explain main features for a horzion chart. Text in
# Spanish"

conclusions_html = f"""
<section class="claves_grafico">
    <h3>Claves para interpretar el gráfico</h3>
    <p>
       El gráfico horizonte muestra la evolución de las emisones por cápita de
       los 15 países con unas emisiones más elevadas totales de CO₂ en el año
       2024. Los países están ordenados en orden descendente en función de sus
       emisiones totales de CO₂ para el año 2024, siendo China el
       país con unas emisiones más elevadas totales seguido por Estados Unidos
       e India. El periodo temporal va desde el año 1950 hasta el 2024. Se ha
       excluido el país Arabia Saudí por falta de algunos datos anuales. El
       gráfico permite comparar las evoluciones de emisiones por cápita anuales,
       donde se pueden destacar trayectorias muy diferenciadas.
    </p>
    <p>
        La mayoría de los países presentados muestran una evolución creciente
        entre el año 1950 y las décadas de 1980 y 1990. A partir de ese punto,
        se puede observar como los países desarrollados (Estados Unidos, Japón,
        Corea del Sur, Canada, Austrialia, Alemania y Rusia) muestran una
        trayectoria descendiente de emisiones de CO₂ per cápita durante las
        últimas décadas aunque siguen teniendo los valores más elevados. Por
        otro lado, los paises en vías de desarrollo (China, India, Indonesia,
        Iran y Turquía) muestran una trayectoria ascendente en este indicador,
        aunque los valores de emisiones de CO₂ per cápita son inferiores al otro
        grupo de países. Brasil, Mexico y Sudáfrica muestran una estabilización
        de sus emisiones de CO₂ per cápita.
    </p>
    <p>
        En el año 2024 los países con mayores emisiones de CO₂ per cápita entre
        los 15 seleccionados fueron: {highest_text} (datos en toneladas de CO₂
        por persona y año). Por otro lado, en el extremo inferior se situaron:
         {lowest_text} (datos en toneladas de CO₂ por persona y año).
    </p>
    <p>
        Entre los años {first_year} y {last_year}, el mayor aumento absoluto
        de emisiones per cápita correspondió a {largest_increase_country}
        ({largest_increase:+.2f} toneladas por persona).
        Por el contrario, el mayor descenso correspondió a
        {largest_decrease_country} ({largest_decrease:+.2f} toneladas por
        persona).
    </p>
</section>
"""

display(HTML("""
<style>
.commentary {
    background: #FFFFFF;
    border-left: 4px solid #64748B;
    border-radius: 5px;
    padding: 12px 18px;
    margin: 18px 0 24px 0;
    color: #263445;
    line-height: 1.6;
    max-width: 1100px;
}
.commentary h3 {
    margin-top: 0;
    margin-bottom: 8px;
    font-size: 18px;
}
.commentary p {
    margin: 8px 0;
}
</style>
""" + conclusions_html))

In [16]:
# Code to prepare the 2024 reference table with the data used in teh
# visualization
table_2024 = (
    df_horizon[df_horizon["year"] == 2024]
    [["year", "country", "iso_code", "co2_per_capita", "co2"]]
    .sort_values("co2", ascending=False)
    .copy()
)

# Code to round the numeric columns for display
table_2024["co2_per_capita"] = table_2024["co2_per_capita"].round(2)
table_2024["co2"] = table_2024["co2"].round(2)

# Code to rename columns for the published page in Spanish
table_2024 = table_2024.rename(columns={
    "year": "Año",
    "country": "País",
    "iso_code": "Código ISO",
    "co2_per_capita": "CO₂ per cápita (t/persona)",
    "co2": "CO₂ total (millones de toneladas)"
})

# Code to convert the table to HTML
table_html = table_2024.to_html(
    index=False,
    classes="data-table",
    border=0,
    escape=True
)

display(table_2024)

,Año,País,Código ISO,CO₂ per cápita (t/persona),CO₂ total (millones de toneladas)
9933,2024,China,CHN,8.66,12289.04
48007,2024,United States,USA,14.20,4904.12
21762,2024,India,IND,2.20,3193.48
38314,2024,Russia,RUS,12.29,1780.52
23812,2024,Japan,JPN,7.77,961.87
21937,2024,Indonesia,IDN,2.87,812.22
22662,2024,Iran,IRN,8.66,792.63
43038,2024,South Korea,KOR,11.29,583.68
18319,2024,Germany,DEU,6.77,572.32
8877,2024,Canada,CAN,13.42,533.34


In [17]:
# Code to convert the Plotly figure to an embeddable HTML fragment
chart_html = fig_emissions.to_html(
    full_html=False,
    include_plotlyjs="cdn",
    config={"responsive": True, "displaylogo": False}
)

# Code to build the complete web page
# Reference 1: ChatGPT Prompt: "Review the html code to display three objects
# and show a correct format"
html_page = f"""
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>Emisiones de CO₂ per cápita para los 15 países con más emisiones en la actualidad entre 1950–2024</title>

    <style>
        body {{
            font-family: Arial, Helvetica, sans-serif;
            max-width: 1250px;
            margin: 0 auto;
            padding: 24px;
            color: #263238;
            background-color: #ffffff;
            line-height: 1.6;
        }}

        h1, h2 {{
            color: #174A7E;
            line-height: 1.25;
        }}

        h1 {{
            margin-bottom: 12px;
        }}

        .intro, .data-section {{
            margin: 28px 0;
        }}

        .chart-container {{
            width: 100%;
            margin: 20px 0;
        }}

        .conclusions {{
            margin: 35px 0;
            padding: 24px 28px;
            background-color: #F3F7FB;
            border: 1px solid #D9E5F0;
            border-left: 5px solid #4388C7;
            border-radius: 8px;
        }}

        .conclusions h2 {{
            margin-top: 0;
            margin-bottom: 16px;
            color: #174A7E;
            font-size: 21px;
        }}

        .conclusions p {{
            margin: 12px 0;
            color: #37474F;
            line-height: 1.7;
        }}

        .conclusions p:last-child {{
            margin-bottom: 0;
        }}

        .data-table-wrapper {{
            width: 100%;
            overflow-x: auto;
        }}

        table.data-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 14px;
        }}

        .data-table th, .data-table td {{
            padding: 10px 12px;
            border-bottom: 1px solid #D9DEE5;
            text-align: left;
        }}

        .data-table th {{
            background-color: #174A7E;
            color: white;
            white-space: nowrap;
        }}

        .data-table tbody tr:nth-child(even) {{
            background-color: #F3F6F9;
        }}

        .source {{
            margin-top: 36px;
            padding-top: 16px;
            border-top: 1px solid #D9DEE5;
            font-size: 13px;
            color: #58636D;
        }}

        .conclusions {{
            margin: 35px 0;
            padding: 24px 28px;
            background-color: #F3F7FB;
            border: 1px solid #D9E5F0;
            border-left: 5px solid #4388C7;
            border-radius: 8px;
        }}

        .conclusions h2 {{
            margin-top: 0;
            margin-bottom: 16px;
            color: #174A7E;
            font-size: 21px;
        }}

        .conclusions p {{
            margin: 12px 0;
            color: #37474F;
            line-height: 1.7;
        }}

        .conclusions p:last-child {{
            margin-bottom: 0;
        }}

        a {{
            color: #174A7E;
        }}

        @media (max-width: 700px) {{
            body {{
                padding: 14px;
            }}

            h1 {{
                font-size: 26px;
            }}

            .data-table th, .data-table td {{
                padding: 8px;
            }}
        }}
    </style>
</head>

<body>

    <header>
        <h1>Emisiones de CO₂ per cápita para los 15 países con más emisiones en la actualidad entre 1950–2024</h1>
    </header>

    <main>
        <section class="chart-container">
            {chart_html}
        </section>

        {conclusions_html}

        <section class="data-section">
            <h2>Datos utilizados en el gráfico</h2>

            <div class="data-table-wrapper">
                {table_html}
            </div>
        </section>
    </main>

    <footer class="source">
        <p>
            <strong>Fuente:</strong>
            Our World in Data — CO₂ and Greenhouse Gas Emissions Data
        </p>
    </footer>

</body>
</html>
"""

# Save the page as an HTML file
output_path = "index.html"

with open(output_path, "w", encoding="utf-8") as file:
    file.write(html_page)

print(f"HTML file created: {output_path}")

HTML file created: index.html


In [18]:
from google.colab import files
files.download("index.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>